# Netflix Movies & TV Shows Data Analysis
### An End-to-End Exploratory Data Analysis & Business Intelligence Project
**Author**: Antigravity Data Analytics  
**Dataset**: Netflix Titles Catalog (5,800+ records)  
**Tools**: Python, Pandas, NumPy, Matplotlib, Seaborn

---

## Project Objective
The goal of this analysis is to conduct a rigorous, end-to-end exploratory data analysis (EDA) of the Netflix Movies and TV Shows catalog to uncover content acquisition trends, international expansion trajectories, genre concentrations, maturity ratings, runtime distributions, and strategic catalog composition patterns.


## 1. Setup & Library Imports


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Ensure module imports work smoothly
sys.path.append("..")
from src.data_cleaning import clean_netflix_pipeline, get_exploded_series, get_movies_df, get_tvshows_df
from src.exploratory_analysis import (
    analyze_content_type, analyze_countries, analyze_genres, 
    analyze_temporal_trends, analyze_ratings_and_demographics,
    analyze_directors, analyze_cast, analyze_movie_durations,
    analyze_tv_shows, analyze_comparative
)
from src.visualization import (
    apply_netflix_theme, NETFLIX_RED, DARK_BG, CARD_BG, TEXT_WHITE,
    plot_01_movies_vs_tvshows, plot_02_top10_countries, plot_03_top10_genres,
    plot_04_content_growth, plot_05_monthly_additions, plot_06_rating_distribution,
    plot_07_top_directors, plot_08_top_actors, plot_09_movie_duration,
    plot_10_tvshow_seasons, plot_11_genre_rating_heatmap, plot_12_movies_vs_tvshows_growth
)

# Apply publication-ready dark styling
apply_netflix_theme()
%matplotlib inline
print("Libraries imported and theme configured successfully.")


---
## 2. Phase 2: Data Understanding & Initial Inspection

In this phase, we load the raw dataset and inspect its structure, dimensions, column data types, missing value percentages, and duplication patterns.


In [ ]:
# Load raw dataset
raw_df = pd.read_csv("../data/netflix_titles.csv")
print(f"Dataset Shape: {raw_df.shape[0]:,} rows and {raw_df.shape[1]} columns\n")
raw_df.head(5)


In [ ]:
# Dataset schema and non-null counts
raw_df.info()


In [ ]:
# Missing value analysis
missing_df = pd.DataFrame({
    "Missing Count": raw_df.isnull().sum(),
    "Missing Percentage (%)": (raw_df.isnull().sum() / len(raw_df) * 100).round(2)
})
missing_df[missing_df["Missing Count"] > 0].sort_values(by="Missing Count", ascending=False)


In [ ]:
# Check duplicate records
exact_dups = raw_df.duplicated().sum()
content_dups = raw_df.duplicated(subset=["title", "type", "release_year"]).sum()
print(f"Exact row duplicates: {exact_dups}")
print(f"Duplicate titles (sharing title, type, release_year): {content_dups}")


---
## 3. Phase 3: Data Cleaning & Feature Engineering

### Cleaning Actions & Justifications:
1. **Handling Missing Values**:
   - `director` (~32.5% missing): Imputed with `'Unknown Director'` (TV shows frequently do not credit single directors; dropping would lose a third of catalog).
   - `cast` (~9.5% missing): Imputed with `'Unknown Cast'` (documentaries, news specials, or unlisted ensembles).
   - `country` (~7.3% missing): Imputed with `'Unknown Country'`.
   - `rating` (10 missing): Imputed as `'Unavailable'`.
2. **Deduplication**: True duplicate releases with matching (title, type, release_year) are pruned, retaining the first occurrence while preserving valid remakes.
3. **Date Parsing**: `date_added` converted to `datetime64[ns]`; derived features: `year_added`, `month_added`, `month_name_added`, `release_to_add_lag`.
4. **Duration Parsing**: Split into numeric `duration_min` for Movies and `seasons` for TV Shows.
5. **Audience Demographics**: Ratings mapped into standard age tiers (`Adults (18+)`, `Teens (13-17)`, `Older Kids (7-12)`, `Little Kids (0-6)`, `Unrated`).
6. **Multi-Value Parsing**: Created `primary_country`, `primary_genre`, `country_count`, and `is_multi_country` indicators.


In [ ]:
# Run the automated cleaning pipeline
df = clean_netflix_pipeline(
    raw_path="../data/netflix_titles.csv",
    output_path="../data/netflix_cleaned.csv"
)
df[["title", "type", "release_year", "year_added", "duration_min", "seasons", "age_group", "primary_country"]].head(5)


---
## 4. Phase 4 & 5: Exploratory Data Analysis & Visualizations

We now systematically address all core analytical questions with statistics, tables, and visualization charts.


### Question 1: What is the distribution of Movies vs TV Shows?


In [ ]:
type_analysis = analyze_content_type(df)
print(type_analysis)
plot_01_movies_vs_tvshows(df, output_dir="../visualizations")
plt.show()


### Question 2: Which countries produce the most Netflix content?


In [ ]:
country_data = analyze_countries(df, top_n=10)
print("Top Content Producing Countries (All Credits):")
print(country_data["top_countries_all_credits"])
print(f"\nPercentage of international co-productions: {country_data['co_production_percentage']}%")
plot_02_top10_countries(df, output_dir="../visualizations")
plt.show()


### Question 3: What are the most common genres / categories?


In [ ]:
genre_data = analyze_genres(df, top_n=10)
print("Top 10 Genres Overall:")
print(genre_data["top_genres_overall"])
plot_03_top10_genres(df, output_dir="../visualizations")
plt.show()


### Question 4: How has Netflix content grown over the years?


In [ ]:
temporal_data = analyze_temporal_trends(df)
print("Annual Catalog Additions (2014 - 2019):")
print(temporal_data["yearly_additions"].tail(6))
plot_04_content_growth(df, output_dir="../visualizations")
plt.show()


### Question 5: What seasonal patterns exist in monthly content additions?


In [ ]:
print("Monthly Content Additions Distribution:")
print(temporal_data["monthly_additions"])
plot_05_monthly_additions(df, output_dir="../visualizations")
plt.show()


### Question 6: Which ratings and maturity demographics dominate Netflix?


In [ ]:
rating_data = analyze_ratings_and_demographics(df)
print("Rating Breakdown by Type:")
print(rating_data["rating_by_type"])
print("\nTarget Demographic Age Groups:")
print(rating_data["demographics"])
plot_06_rating_distribution(df, output_dir="../visualizations")
plt.show()


### Question 7: Which directors have created the most content?


In [ ]:
director_data = analyze_directors(df, top_n=10)
print("Top 10 Directors on Netflix:")
print(director_data["top_directors_overall"])
plot_07_top_directors(df, output_dir="../visualizations")
plt.show()


### Question 8: Which actors appear most frequently across titles?


In [ ]:
cast_data = analyze_cast(df, top_n=10)
print("Top 10 Most Frequently Cast Actors Globally:")
print(cast_data["top_actors_global"])
print("\nTop 5 Actors in India:")
print(cast_data["top_actors_india"].head(5))
print("\nTop 5 Actors in United States:")
print(cast_data["top_actors_us"].head(5))
plot_08_top_actors(df, output_dir="../visualizations")
plt.show()


### Question 9: What is the distribution of movie durations?


In [ ]:
duration_data = analyze_movie_durations(df)
print("Movie Runtime Summary Statistics (Minutes):")
for k, v in duration_data["duration_stats"].items():
    print(f"  {k:<15}: {v}")
print("\nLongest Movies:")
print(duration_data["longest_movies"][["title", "release_year", "duration_min"]])
print("\nShortest Movies:")
print(duration_data["shortest_movies"][["title", "release_year", "duration_min"]])
plot_09_movie_duration(df, output_dir="../visualizations")
plt.show()


### Question 10: What is the distribution of TV Show seasons?


In [ ]:
tv_data = analyze_tv_shows(df)
print(f"Single-Season Shows: {tv_data['single_season_pct']}%")
print(f"Two-Season Shows: {tv_data['two_season_pct']}%")
print(f"Three or More Seasons: {tv_data['three_plus_seasons_pct']}%")
print("\nLongest Running Shows:")
print(tv_data["longest_running_shows"][["title", "seasons", "primary_country"]])
plot_10_tvshow_seasons(df, output_dir="../visualizations")
plt.show()


### Question 11: How do genres correlate with target audience demographics?


In [ ]:
plot_11_genre_rating_heatmap(df, output_dir="../visualizations")
plt.show()


### Question 12: How do Movies and TV Shows compare in growth and across countries?


In [ ]:
comp_data = analyze_comparative(df)
print("Content Distribution Across Top 10 Producing Countries:")
print(comp_data["country_vs_type"])
plot_12_movies_vs_tvshows_growth(df, output_dir="../visualizations")
plt.show()


---
## 5. Phase 6: Executive Strategic Insights & Recommendations

### 1. Catalog Composition & Retention Strategy
- **Observation**: Movies make up **67.5%** of the catalog, while TV Shows comprise **32.5%**. However, TV series additions accelerated rapidly from 2016 onwards.
- **Business Implication**: While films attract initial subscribers with star power and brand recognition, episodic television drives sustained weekly engagement, reducing churn. Netflix should continue investing heavily in multi-season episodic intellectual property (IP).

### 2. Geographic Expansion & Local Content Moats
- **Observation**: The United States produces **41.5%** of the catalog, followed by India (**12.9%**) and the United Kingdom (**9.6%**). Over **14.5%** of titles are international co-productions.
- **Business Implication**: Developing local-language originals in high-growth markets (India, Latin America, East Asia) serves a dual purpose: capturing domestic subscriptions and creating cross-border breakout hits (e.g., *Money Heist*, *Squid Game*).

### 3. Mature Audience Skew
- **Observation**: Over **71.9%** of the catalog is classified for mature audiences (**40.7% Adults 18+**, **31.2% Teens 13-17**). Little kids content comprises less than 10%.
- **Business Implication**: Netflix has firmly established itself as a premier adult entertainment destination. To better compete against Disney+ in the high-LTV household family segment, targeted acquisition of evergreen children's animation is recommended.

### 4. Optimal Movie Runtime Window
- **Observation**: The median movie runtime is **97.0 minutes**, with 50% of movies falling squarely between **85 and 113 minutes**.
- **Business Implication**: Audiences strongly favor tight, 90–100 minute narratives for evening home streaming, avoiding the fatigue of 150+ minute theatrical epics.

### 5. Ingestion Seasonality & Campaign Scheduling
- **Observation**: Content additions peak sharply in **January (9.9%)**, **October (9.5%)**, **November (9.7%)**, and **December (8.7%)**.
- **Business Implication**: Catalog refreshes are timed with holiday vacations, winter indoor viewing peaks, and Q4 awards pushes. Marketing spend should mirror this seasonal cadence.
